<a href="https://www.kaggle.com/code/vladimir663/notebook8dbd561641?scriptVersionId=313991558" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import pickle
import os

In [3]:
import os
import pickle
import pandas as pd

KAGGLE_INPUT = '/kaggle/input/datasets/vladimir663/wtc11-with-sqanti3/'

df_model = pd.read_csv(f'{KAGGLE_INPUT}/df_model.csv')
df_orig  = pd.read_csv(f'{KAGGLE_INPUT}/classification_with_orf.tsv', sep='\t')

with open(f'{KAGGLE_INPUT}/sequences.pkl', 'rb') as f:
    seqs = pickle.load(f)

df = df_orig[['isoform']].copy()
df['sequence'] = df['isoform'].map(seqs)
df['target']   = df_model['target_4class'].values
df = df.dropna(subset=['sequence'])

print(f"Dataset: {len(df):,} isoforms")
print(df['target'].value_counts())

Dataset: 489,545 isoforms
target
real_high    202421
uncertain    175559
artifact      66905
real_mid      44660
Name: count, dtype: int64


In [4]:
NUCLEOTIDE_MAP = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
MAX_LEN = 8000

def one_hot(seq, max_len=MAX_LEN):
    seq = seq[:max_len]
    seq_len = len(seq)
    
    seq_bytes = np.frombuffer(seq.encode('ascii'), dtype=np.uint8)
    
    a_mask = (seq_bytes == 65)
    c_mask = (seq_bytes == 67)
    g_mask = (seq_bytes == 71)
    t_mask = (seq_bytes == 84)
    
    encoded = np.stack([a_mask, c_mask, g_mask, t_mask], axis=0).astype(np.float32)
    
    tensor = torch.zeros(4, max_len, dtype=torch.float32)
    tensor[:, :seq_len] = torch.from_numpy(encoded)
    
    return tensor

In [5]:
LABEL_MAP = {
    'real_high': 0,
    'real_mid':  1,
    'uncertain': 2,
    'artifact':  3
}

class IsoformDataset(Dataset):
    def __init__(self, sequences, labels, max_len=MAX_LEN):
        self.sequences = sequences
        self.labels = [LABEL_MAP[l] for l in labels]
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = one_hot(self.sequences[idx], self.max_len)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq, label

In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, 
                                    stratify=df['target'], 
                                    random_state=42)
train_df, val_df  = train_test_split(train_df, test_size=0.1,
                                    stratify=train_df['target'],
                                    random_state=42)

train_ds = IsoformDataset(train_df['sequence'].values, train_df['target'].values)
val_ds   = IsoformDataset(val_df['sequence'].values,   val_df['target'].values)
test_ds  = IsoformDataset(test_df['sequence'].values,  test_df['target'].values)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=128, shuffle=False, num_workers=4)

print(f"Train: {len(train_ds):,}")
print(f"Val:   {len(val_ds):,}")
print(f"Test:  {len(test_ds):,}")

seq_batch, label_batch = next(iter(train_loader))
print(f"\nseq shape:   {seq_batch.shape}")
print(f"label shape: {label_batch.shape}")

Train: 352,472
Val:   39,164
Test:  97,909

seq shape:   torch.Size([128, 4, 8000])
label shape: torch.Size([128])


In [7]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel = 7, dilation = 1):
        super().__init__()
        padding = (kernel - 1) * dilation // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel, dilation=dilation, padding=padding)
        self.bn = nn.BatchNorm1d(out_ch)
        self.dp = nn.Dropout(0.1)

    def forward(self, x):
        return self.dp(F.relu(self.bn(self.conv(x))))

In [10]:
class IsoformClassifier(nn.Module):
    def __init__(self, n_classes = 4):
        super().__init__()
        self.conv1 = ConvBlock(4, 64, kernel = 7, dilation = 1)
        self.conv2 = ConvBlock(64, 64, kernel = 7, dilation = 2)
        self.conv3 = ConvBlock(64, 128, kernel = 7, dilation = 4)
        self.conv4 = ConvBlock(128, 128, kernel = 7, dilation = 8)

        self.classifier = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        x = self.conv4(self.conv3(self.conv2(self.conv1(x))))
        x = x.max(dim = -1).values
        return self.classifier(x)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"GPUs: {torch.cuda.device_count()}")

model = IsoformClassifier(n_classes=4)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Device: {device}")
print(f"Parameters: {total_params:,}")

with torch.no_grad():
    out = model(seq_batch.to(device))
    print(f"Output shape: {out.shape}")  

GPUs: 2
Device: cuda
Parameters: 220,676
Output shape: torch.Size([128, 4])


In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(['artifact', 'real_high', 'real_mid', 'uncertain'])
weights = compute_class_weight('balanced', classes = classes, y = df['target'].values)
class_weights = torch.tensor(weights, dtype = torch.float32).to(device)

In [14]:
device = torch.device('cuda:0')

model = IsoformClassifier(n_classes=4).to(device)

criterion = nn.CrossEntropyLoss(weight = class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer = optimizer, patience = 4, factor = 0.5)

from torch.amp import autocast, GradScaler

scaler = GradScaler('cuda')

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for seq, label in loader:
        seq, label = seq.to(device), label.to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            out = model(seq)
            loss = criterion(out, label)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct += (out.argmax(dim=1) == label).sum().item()
        total += label.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for seq, label in loader:
            seq, label = seq.to(device), label.to(device)
            with autocast('cuda'):
                out = model(seq)
                loss = criterion(out, label)
            total_loss += loss.item()
            correct += (out.argmax(dim=1) == label).sum().item()
            total += label.size(0)
    return total_loss / len(loader), correct / total

In [15]:
EPOCHS = 20
best_val_loss = float('inf')
history = {'train_loss': [], 'val_loss': [], 
        'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step(val_loss)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/working/best_model.pt')
    
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train loss: {train_loss:.4f} acc: {train_acc:.4f} | "
        f"Val loss: {val_loss:.4f} acc: {val_acc:.4f}")

Epoch 01/20 | Train loss: 0.7910 acc: 0.5756 | Val loss: 0.7424 acc: 0.6100
Epoch 02/20 | Train loss: 0.7400 acc: 0.6112 | Val loss: 0.7173 acc: 0.6138
Epoch 03/20 | Train loss: 0.7227 acc: 0.6197 | Val loss: 0.7002 acc: 0.6356
Epoch 04/20 | Train loss: 0.7116 acc: 0.6259 | Val loss: 0.6990 acc: 0.6382


KeyboardInterrupt: 

In [19]:

torch.save(model.state_dict(), '/kaggle/working/conv_only_checkpoint.pt')

In [ ]:
#torch.cuda.empty_cache()